In [1]:
import sys
import tensorflow as tf
import numpy as np

print("Python:", sys.executable)
print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)

Python: C:\Users\shrij\miniconda3\python.exe
TensorFlow: 2.21.0
NumPy: 2.5.3


In [2]:
text_path = tf.keras.utils.get_file(
    'shakespeare.txt',
    'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
)

text = open(text_path, 'r', encoding='utf-8').read().lower()

print("Dataset loaded successfully!")
print("Total characters:", len(text))
print("\nFirst 500 characters:\n")
print(text[:500])

1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Dataset loaded successfully!
Total characters: 1115394

First 500 characters:

first citizen:
before we proceed any further, hear me speak.

all:
speak, speak.

first citizen:
you are all resolved rather to die than to famish?

all:
resolved. resolved.

first citizen:
first, you know caius marcius is chief enemy to the people.

all:
we know't, we know't.

first citizen:
let us kill him, and we'll have corn at our own price.
is't a verdict?

all:
no more talking on't; let it be done: away, away!

second citizen:
one word, good citizens.

first citizen:
we are accounted poor


In [3]:
chars = sorted(set(text))

c2i = {c: i for i, c in enumerate(chars)}
i2c = {i: c for i, c in enumerate(chars)}

print("Number of unique characters:", len(chars))
print("\nCharacters:")
print(chars)

Number of unique characters: 39

Characters:
['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [4]:
seq_len = 40

X = []
y = []

for i in range(len(text) - seq_len):
    input_seq = text[i:i + seq_len]
    target_char = text[i + seq_len]

    X.append([c2i[c] for c in input_seq])
    y.append(c2i[target_char])

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1115354, 40)
y shape: (1115354,)


In [6]:
model = tf.keras.Sequential([
    tf.keras.Input(shape=(seq_len,), dtype=tf.int32),
    tf.keras.layers.Embedding(len(chars), 64),
    tf.keras.layers.LSTM(128),
    tf.keras.layers.Dense(len(chars), activation='softmax')
])

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam'
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━┓
┃                  ┃ Output      ┃  Param ┃
┃ Layer (type)     ┃ Shape       ┃      # ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━┩
│ embedding_1      │ (None, 40,  │  2,496 │
│ (Embedding)      │ 64)         │        │
├──────────────────┼─────────────┼────────┤
│ lstm_1 (LSTM)    │ (None, 128) │ 98,816 │
├──────────────────┼─────────────┼────────┤
│ dense_1 (Dense)  │ (None, 39)  │  5,031 │
└──────────────────┴─────────────┴────────┘

 Total params: 106,343 (415.40 KB)

 Trainable params: 106,343 (415.40 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
history = model.fit(
    X,
    y,
    batch_size=128,
    epochs=1
)

8714/8714 ━━━━━━━━━━━━━━━━━━━━ 912s 104ms/step - loss: 1.8873


In [9]:
def generate(seed, length=300):
    seed = seed.lower()

    seq = [c2i[c] for c in seed]

    for _ in range(length):
        inp = np.array(seq[-seq_len:]).reshape(1, -1)

        pred = model.predict(inp, verbose=0)[0]

        next_idx = np.random.choice(len(pred), p=pred)

        seq.append(next_idx)

    return seed + ''.join(i2c[i] for i in seq[len(seed):])

In [10]:
seed = "shall i compare thee to a summer's day?\n"

generated_text = generate(seed, length=300)

print("\nGenerated Text:\n")
print(generated_text)


Generated Text:

shall i compare thee to a summer's day?

menent:
but lew who have canseadong, everghore. 'cils sham!
we fowt's no pying tome that know not the tots me,
the frair him, then, to marry is rightatisire,
frame the gregh'd will to that petith a thenf:
i thou beat, tham with the to an him.

commomes:
o full nother, but the courses ontony.

a
coo


In [11]:
print("Generated Text Evaluation")
print("-------------------------")
print("Seed prompt:", seed)
print("Generated length:", len(generated_text))
print("Training epochs: 1")
print("Final training loss: 1.8873")

print("\nFluency:")
print("The generated text contains English-like words, punctuation,")
print("spaces, and Shakespearean-style sentence structures.")

print("\nCoherence:")
print("The output shows partial coherence and stylistic similarity")
print("to the Shakespeare corpus, although some words are incorrect.")

Generated Text Evaluation
-------------------------
Seed prompt: shall i compare thee to a summer's day?

Generated length: 340
Training epochs: 1
Final training loss: 1.8873

Fluency:
The generated text contains English-like words, punctuation,
spaces, and Shakespearean-style sentence structures.

Coherence:
The output shows partial coherence and stylistic similarity
to the Shakespeare corpus, although some words are incorrect.


In [12]:
seed2 = "to be, or not to be, that is the question:\n"

generated_text2 = generate(seed2, length=300)

print("\nGenerated Text:\n")
print(generated_text2)


Generated Text:

to be, or not to be, that is the question:
to the take of fead, hot make you we srrit
save a kinglestervace;
is and the bapter i'llage in the would
with neveren you conten hantadions sween.
then farter, thoust jour, this do your slame hourm?

corsicenter:
my righed dight for this mary fellemio,
and hear he for theaved up it'tis
of the distim
